In [7]:
import os
import warnings

import numpy as np
import pandas as pd

from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
)

warnings.filterwarnings("ignore", category=ConvergenceWarning)

LABEL_COL = "nonlinear"
TEST_SIZE = 0.2
N_SPLITS = 5
RANDOM_STATE = 42
CSV_PATH = os.path.join(os.path.dirname(os.getcwd()), "data", "data.csv")

np.random.seed(RANDOM_STATE)

In [8]:
# Load pre-engineered features data
df = pd.read_csv(CSV_PATH)

assert LABEL_COL in df.columns, f"Missing label column: {LABEL_COL}"
assert np.isin(df[LABEL_COL], [0, 1]).all(), "Labels must be binary: 0 (linear), 1 (nonlinear)."

y = df[LABEL_COL].to_numpy(dtype=int)

class_counts = pd.Series(y, name=LABEL_COL).value_counts().sort_index()
print("Class distribution (0=linear, 1=nonlinear):")
print(class_counts)
print(f"\nDataset shape: {df.shape}")
print(f"Feature columns: {list(df.columns[:-1])}")

# Display feature summary statistics by class
features_df = df.drop(columns=[LABEL_COL])

print("\nMean feature value by class (0=linear, 1=nonlinear):")
print(features_df.assign(nonlinear=y).groupby("nonlinear").mean())

print("\nStandard deviation by class:")
print(features_df.assign(nonlinear=y).groupby("nonlinear").std())

Class distribution (0=linear, 1=nonlinear):
nonlinear
0    15000
1    15000
Name: count, dtype: int64

Dataset shape: (30000, 8)
Feature columns: ['crest_factor', 'envelope_skewness', 'envelope_kurtosis', 'peak_power', 'normalized_m4', 'spectral_flatness', 'spectral_entropy']

Mean feature value by class (0=linear, 1=nonlinear):
           crest_factor  envelope_skewness  envelope_kurtosis  peak_power  \
nonlinear                                                                   
0              3.775349           0.717540           3.078003   12.235643   
1              3.782529           0.720697           3.085032   12.193696   

           normalized_m4  spectral_flatness  spectral_entropy  
nonlinear                                                      
0                2.53538           0.298240          8.640526  
1                2.54025           0.301588          8.662930  

Standard deviation by class:
           crest_factor  envelope_skewness  envelope_kurtosis  peak_power 

In [9]:
# Prepare features and train/test split
X = df.drop(columns=[LABEL_COL])
y_target = y

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_target,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_target,
)

# Scale data without leakage: fit on train only, transform train/test with same scaler
scaler = StandardScaler()
X_train = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index,
)
X_test = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index,
)

# Initialize model results table
results_df = pd.DataFrame(
    columns=[
        "model",
        "train_accuracy",
        "accuracy_mean",
        "precision_mean",
        "recall_mean",
        "f1_mean",
        "auc_mean",
    ]
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train class balance: {np.bincount(y_train)}")
print(f"Test class balance: {np.bincount(y_test)}")

Train shape: (24000, 7), Test shape: (6000, 7)
Train class balance: [12000 12000]
Test class balance: [3000 3000]


In [13]:
# SVM

param_grid = {
    "C": np.logspace(-3, 3, 7),
    "gamma": np.logspace(-4, 1, 6),
}

svm_base = SVC(kernel="rbf", random_state=RANDOM_STATE)

random_search = RandomizedSearchCV(
    estimator=svm_base,
    param_distributions=param_grid,
    n_iter=10,
    scoring="roc_auc",
    cv=StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE),
    n_jobs=-1,
    refit=True,
    verbose=1,
    random_state=RANDOM_STATE
)

random_search.fit(X_train, y_train)

svm_model = random_search.best_estimator_
print(f"Best SVM params: {random_search.best_params_}")

train_accuracy = accuracy_score(y_train, svm_model.predict(X_train))

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []
auc_scores = []

for train_idx, val_idx in cv.split(X_train, y_train):
    X_fold_train = X_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx]
    y_fold_train = y_train[train_idx]
    y_fold_val = y_train[val_idx]

    fold_model = clone(svm_model)
    fold_model.fit(X_fold_train, y_fold_train)

    y_val_pred = fold_model.predict(X_fold_val)
    y_val_score = fold_model.decision_function(X_fold_val)

    accuracy_scores.append(accuracy_score(y_fold_val, y_val_pred))
    precision_scores.append(precision_score(y_fold_val, y_val_pred, zero_division=0))
    recall_scores.append(recall_score(y_fold_val, y_val_pred, zero_division=0))
    f1_scores.append(f1_score(y_fold_val, y_val_pred, zero_division=0))
    auc_scores.append(roc_auc_score(y_fold_val, y_val_score))

svm_row = {
    "model": "SVM",
    "train_accuracy": train_accuracy,
    "accuracy_mean": float(np.mean(accuracy_scores)),
    "precision_mean": float(np.mean(precision_scores)),
    "recall_mean": float(np.mean(recall_scores)),
    "f1_mean": float(np.mean(f1_scores)),
    "auc_mean": float(np.mean(auc_scores)),
}

results_df.loc[len(results_df)] = svm_row
print(results_df)

Best SVM params: {'gamma': np.float64(0.01), 'C': np.float64(10.0)}
           model  train_accuracy  accuracy_mean  precision_mean  recall_mean  \
0            SVM        0.592292       0.588958        0.590827     0.578500   
1  Random Forest        0.668333       0.548500        0.549440     0.539417   
2            SVM        0.592292       0.588958        0.590827     0.578500   

    f1_mean  auc_mean  
0  0.584583  0.626560  
1  0.544367  0.562216  
2  0.584583  0.626560  


In [15]:
# Random Forest

param_grid = {
    "n_estimators": [100, 300, 500],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

rf_base = RandomForestClassifier(random_state=RANDOM_STATE)

random_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_grid,
    n_iter=10,
    scoring="roc_auc",
    cv=StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE),
    n_jobs=-1,
    refit=True,
    verbose=1,
    random_state=RANDOM_STATE,
)

random_search.fit(X_train, y_train)

rf_model = random_search.best_estimator_
print(f"Best RF params: {random_search.best_params_}")

train_accuracy = accuracy_score(y_train, rf_model.predict(X_train))

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []
auc_scores = []

for train_idx, val_idx in cv.split(X_train, y_train):
    X_fold_train = X_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx]
    y_fold_train = y_train[train_idx]
    y_fold_val = y_train[val_idx]

    fold_model = clone(rf_model)
    fold_model.fit(X_fold_train, y_fold_train)

    y_val_pred = fold_model.predict(X_fold_val)
    y_val_score = fold_model.predict_proba(X_fold_val)[:, 1]

    accuracy_scores.append(accuracy_score(y_fold_val, y_val_pred))
    precision_scores.append(precision_score(y_fold_val, y_val_pred, zero_division=0))
    recall_scores.append(recall_score(y_fold_val, y_val_pred, zero_division=0))
    f1_scores.append(f1_score(y_fold_val, y_val_pred, zero_division=0))
    auc_scores.append(roc_auc_score(y_fold_val, y_val_score))

rf_row = {
    "model": "Random Forest",
    "train_accuracy": train_accuracy,
    "accuracy_mean": float(np.mean(accuracy_scores)),
    "precision_mean": float(np.mean(precision_scores)),
    "recall_mean": float(np.mean(recall_scores)),
    "f1_mean": float(np.mean(f1_scores)),
    "auc_mean": float(np.mean(auc_scores)),
}

results_df.loc[len(results_df)] = rf_row
print(results_df)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best RF params: {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_depth': 10}
           model  train_accuracy  accuracy_mean  precision_mean  recall_mean  \
0            SVM        0.592292       0.588958        0.590827     0.578500   
1  Random Forest        0.668333       0.548500        0.549440     0.539417   
2            SVM        0.592292       0.588958        0.590827     0.578500   
3  Random Forest        0.668333       0.548500        0.549440     0.539417   

    f1_mean  auc_mean  
0  0.584583  0.626560  
1  0.544367  0.562216  
2  0.584583  0.626560  
3  0.544367  0.562216  
